# Convert `.pth` to `.pte`

This notebook loads the trained DenseNet121 checkpoint and exports it to an ExecuTorch `.pte` file.

Expected input: `models/densenet121_brain_tumor.pth`

Expected output: `models/densenet121_brain_tumor.pte`


In [1]:
import os
# To avoid some CUDA symbol issues during export if present
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

from pathlib import Path

try:
    import torch
    import torch.nn as nn
    from torchvision import models
    from executorch.exir import to_edge
except ImportError as exc:
    raise ImportError(
        "Missing dependencies. Please ensure `torch`, `torchvision`, and `executorch` are installed."
    ) from exc

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing a models/ directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
MODELS_DIR = PROJECT_ROOT / "models"
PTH_MODEL_PATH = MODELS_DIR / "densenet121_brain_tumor.pth"
PTE_MODEL_PATH = MODELS_DIR / "densenet121_brain_tumor.pte"

print("Project root:", PROJECT_ROOT)
print("Input checkpoint:", PTH_MODEL_PATH)
print("Output program:", PTE_MODEL_PATH)


Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu126 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Project root: /media/tst_imperial/Projects/Brain_tumor/Brain_tu
Input checkpoint: /media/tst_imperial/Projects/Brain_tumor/Brain_tu/models/densenet121_brain_tumor.pth
Output program: /media/tst_imperial/Projects/Brain_tumor/Brain_tu/models/densenet121_brain_tumor.pte


In [2]:
def build_model() -> torch.nn.Module:
    model = models.densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, 4)

    state_dict = torch.load(PTH_MODEL_PATH, map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    return model


model = build_model()
example_inputs = (torch.randn(1, 3, 224, 224),)

with torch.no_grad():
    eager_output = model(*example_inputs)

print("Checkpoint loaded successfully.")
print("Example output shape:", tuple(eager_output.shape))


Checkpoint loaded successfully.
Example output shape: (1, 4)


In [3]:
exported_program = torch.export.export(model, example_inputs)
edge_program = to_edge(exported_program)
executorch_program = edge_program.to_executorch()

PTE_MODEL_PATH.write_bytes(executorch_program.buffer)

print(f"Saved ExecuTorch program to: {PTE_MODEL_PATH}")
print(f"File size: {PTE_MODEL_PATH.stat().st_size / 1e6:.2f} MB")


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/media/tst_imperial/Projects/Brain_tumor/Brain_tu/venv/lib/python3.12/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


Saved ExecuTorch program to: /media/tst_imperial/Projects/Brain_tumor/Brain_tu/models/densenet121_brain_tumor.pte
File size: 28.36 MB
